In [4]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/ingest.py
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-08 06:14:50--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-08 06:14:51 (17.2 MB/s) - ‘ingest.py’ saved [738/738]

--2026-07-08 06:14:51--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json
user_prompt = json.dumps(doc)

In [9]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
response.output_parsed.questions

['Can I still join the course if I found it late, and is there anything special I need to do to get a certificate?',
 'If I join after the course has already started, can I still earn a certificate or do I need to do something before a deadline?',
 'I just came across this course — is it too late to join, and how do I make sure I’m still eligible for the certificate?',
 'Am I allowed to enroll now even though I discovered the course late, and does the project have to be submitted while submissions are open for the certificate?',
 'If I start the course late, will I still be able to get a certificate as long as I submit the project on time?']

In [13]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — is it still possible to join and start learning now?', 'If I enroll after the course has already started, can I still get the certificate?', 'What do I need to do to be eligible for a certificate if I join late?', 'Is there a deadline for submitting the project if I want the course certificate?', 'Can I take the course now even if I missed the initial start date, and will late enrollment affect certification?']


In [16]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=312)

In [17]:
from evaluation_utils import calc_price

In [18]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00047250000000000005,
 'total_cost': 0.00062775}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — is it still possible to join and start learning now?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll after the course has already started, can I still get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for a certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project if I want the course certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I take the course now even if I missed the initial start date, and will late enrollment affect certification?',
  'document': '74eb249bbf'}]

In [20]:
import pandas as pd

In [21]:
pd.DataFrame(records)

,question,document
0,I just found this course late — is it still po...,74eb249bbf
1,If I enroll after the course has already start...,74eb249bbf
2,What do I need to do to be eligible for a cert...,74eb249bbf
3,Is there a deadline for submitting the project...,74eb249bbf
4,Can I take the course now even if I missed the...,74eb249bbf


In [22]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
generate_ground_truth(doc)

([{'question': 'I just found this course late — can I still sign up and join in?',
   'document': '74eb249bbf'},
  {'question': 'If I join after the course started, is that okay or did I miss my chance?',
   'document': '74eb249bbf'},
  {'question': 'Can latecomers still take part in the course, or is it too late now?',
   'document': '74eb249bbf'},
  {'question': 'I’m new here — can I still join the course and do the project later?',
   'document': '74eb249bbf'},
  {'question': 'Is it still possible to join this course now, and what do I need to do to get a certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=314))

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [27]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [28]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [30]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [31]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07749975000000002

In [32]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07749975000000002

In [33]:
df_ground_truth = pd.DataFrame(ground_truth)

In [35]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [36]:
len(df_ground_truth)

515